---
### Imports
---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

---
### Loading Dataset
---

In [2]:
df = pd.read_csv("../data/processed/california_housing_clean.csv")

---
### Prepare Train/Test Data
---

In [3]:
# Seperate labels
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

# Split test/train sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# List to store models metrics
results = []

---
### Baseline Model: Linear Regression
---

In [4]:
# Imports
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Define model
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

# Metrics
rmse_lin = root_mean_squared_error(y_test, y_pred_lin)
mae_lin = mean_absolute_error(y_test, y_pred_lin)
r2_lin = r2_score(y_test, y_pred_lin)

# Append results
results.append({
    "Model": "Linear Regression", 
    "RMSE": rmse_lin, 
    "MAE": mae_lin,
    "R²": r2_lin
})

In [5]:
results

[{'Model': 'Linear Regression',
  'RMSE': 72303.08630781871,
  'MAE': 50841.25503931168,
  'R²': 0.6010607087984375}]

---
### Decision Tree Regressor
---

In [6]:
# Imports
from sklearn.tree import DecisionTreeRegressor

# Define model
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train, y_train)
y_pred_tree = tree_reg.predict(X_test)

# Metrics
rmse_tree = root_mean_squared_error(y_test, y_pred_tree)
mae_tree = mean_absolute_error(y_test, y_pred_tree)
r2_tree = r2_score(y_test, y_pred_tree)

# Append results
results.append({
    "Model": "Decision Tree", 
    "RMSE": rmse_tree, 
    "MAE": mae_tree,
    "R²": r2_tree
})

---
### Random Forest Regressor
---

In [7]:
# Imports
from sklearn.ensemble import RandomForestRegressor

# Define model
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

# Metrics
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# Append results
results.append({
    "Model": "Random Forest",
    "RMSE": rmse_rf,
    "MAE": mae_rf,
    "R²": r2_rf
})

---
### XGBoost
---

In [8]:
# Imports
from xgboost import XGBRegressor

# Define model
xgb_reg = XGBRegressor(n_estimators=200, random_state=42)
xgb_reg.fit(X_train, y_train)
y_pred_xgb = xgb_reg.predict(X_test)

# Metrics
rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

# Append results
results.append({
    "Model": "XGBoost",
    "RMSE": rmse_xgb,
    "MAE": mae_xgb,
    "R²": r2_xgb
})


---
### Hyperparameter Tuning
---

In [9]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_reg = XGBRegressor(random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_reg,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_
print("Best Params:", random_search.best_params_)

y_pred_best = best_model.predict(X_test)

rmse_best = root_mean_squared_error(y_test, y_pred_best)
mae_best = mean_absolute_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

results.append({
    "Model": "XGBoost Tuned",
    "RMSE": rmse_best,
    "MAE": mae_best,
    "R²": r2_best
})

print("Best Parameters:", random_search.best_params_)
print("Best Estimator:", random_search.best_estimator_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=3, n_estimators=300, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=3, n_estimators=300, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=3, n_estimators=300, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=3, n_estimators=300, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=3, n_estimators=300, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=300, subsample=0.8; total time=   2.8s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=300, subsample=0.8; total time=   3.3s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=300, subsample=0.8; total time=   3.3s
[CV] END colsam

---
### Compare Results
---

In [10]:
results_df = pd.DataFrame(results)
results_df.sort_values(by="RMSE", ignore_index=True)

,Model,RMSE,MAE,R²
0,XGBoost Tuned,44861.280347,29221.996456,0.846419
1,XGBoost,46822.302905,30839.445863,0.832699
2,Random Forest,50510.098226,32423.715407,0.805307
3,Linear Regression,72303.086308,50841.255039,0.601061
4,Decision Tree,72594.271155,44224.314922,0.597841


---
### Finalize Model
- XGBoost Tuned (RMSE ~ 44K, ~6K better than Random Forest)

###
---